# Day 1: Exploratory Data Analysis & Feature Engineering
## Accident Hotspot Prediction and Road Safety Intelligence System

**Dataset:** US Accidents (2016 - 2023) by Sobhan Moosavi  
**Key Note:** The `Severity` scale represents **traffic delay impact** (1 = short delay, 4 = long disruption), **not fatality or injury severity**.

### Notebook Objectives:
1. Inspect dataset dimensions, data types, and missing value profiles.
2. Remove duplicates and validate geospatial boundaries (continental US).
3. Convert timestamps and engineer temporal indicators (`Hour`, `Day_OfWeek`, `Is_Rush_Hour`, `Is_Weekend`).
4. Analyze severity distribution and examine class imbalance.
5. Explore weather conditions and infrastructure feature prevalence.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure display
pd.set_option('display.max_columns', 50)
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

In [ ]:
# 1. Load Processed Dataset
data_path = Path('../data/processed/cleaned_accidents.csv')
if not data_path.exists():
    data_path = Path('data/processed/cleaned_accidents.csv')

df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} accident records with {df.shape[1]} columns.")
df.head(3)

In [ ]:
# 2. Inspect Target Variable: Traffic Delay Severity
print("Severity Frequency Distribution:")
print(df['Severity'].value_counts().sort_index())
print("\nNormalized Class Proportions:")
print(df['Severity'].value_counts(normalize=True).sort_index().round(4))

In [ ]:
# Plot Severity Distribution
plt.figure(figsize=(7, 4))
colors = ['#38bdf8', '#3b82f6', '#f59e0b', '#ef4444']
counts = df['Severity'].value_counts().sort_index()
plt.bar([f'Severity {k}' for k in counts.index], counts.values, color=colors)
plt.title('Distribution of Accident Severity (Traffic Delay Scale)')
plt.ylabel('Number of Accidents')
plt.xlabel('Severity Category')
plt.show()

In [ ]:
# 3. Temporal Distribution: Commute Rush Hour Impact
hourly_accidents = df.groupby('Hour')['ID'].count()

plt.figure(figsize=(10, 4))
plt.plot(hourly_accidents.index, hourly_accidents.values, marker='o', color='#3b82f6', linewidth=2)
plt.axvspan(7, 9, color='#fbbf24', alpha=0.3, label='Morning Rush (7-9 AM)')
plt.axvspan(16, 18, color='#f87171', alpha=0.3, label='Evening Rush (4-6 PM)')
plt.title('Accident Frequency by Hour of Day')
plt.xlabel('Hour (00:00 - 23:00)')
plt.ylabel('Accident Count')
plt.xticks(range(0, 24))
plt.legend()
plt.show()

In [ ]:
# 4. Infrastructure Features Breakdown
infra_cols = ['Junction', 'Crossing', 'Traffic_Signal', 'Stop']
infra_pcts = {col: df[col].mean() * 100 for col in infra_cols}

plt.figure(figsize=(8, 4))
plt.bar(infra_pcts.keys(), infra_pcts.values(), color='#10b981')
plt.title('Prevalence of Road Infrastructure Features in Accidents')
plt.ylabel('Percentage of Accidents (%)')
for col, val in infra_pcts.items():
    plt.text(col, val + 0.5, f"{val:.1f}%", ha='center', fontweight='bold')
plt.ylim(0, max(infra_pcts.values()) + 5)
plt.show()